# 03 Build Semantic Model

This notebook transforms the raw synthetic tables into a clean semantic model for analytics, forecasting, and later GenAI-generated insights.

The notebook covers two stages:

-  build cleaned dimensions
-  build cleaned facts

## Input files

This notebook reads the following raw CSV files from `data/raw/`:

- `reports.csv`
- `users.csv`
- `report_pages.csv`
- `dates.csv`
- `report_views.csv`
- `report_page_views.csv`
- `report_load_times.csv`

## Output files

This notebook writes the following processed CSV files to `data/processed/`:

### Dimensions
- `dim_date.csv`
- `dim_user.csv`
- `dim_report.csv`
- `dim_page.csv`

### Facts
- `fact_report_views.csv`
- `fact_page_views.csv`
- `fact_report_loads.csv`

## Why this notebook matters

We are moving from:

`raw source-aligned tables -> clean star schema`

That semantic model will become the foundation for:

- behavioral analysis
- forecasting
- model evaluation
- later GenAI insight generation


## Setup

The notebook is expected to live inside the `notebooks/` folder, so we move one level up to find the project root and then define the raw and processed data folders.


In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
RAW_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw path:", RAW_PATH)
print("Processed path:", PROCESSED_PATH)

Project root: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting
Raw path: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/raw
Processed path: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed


## Load raw tables

We load the raw CSV files created in the previous notebook.

These are still source-aligned tables. They are useful for simulation, but not yet ideal for downstream analytics.


In [2]:
reports = pd.read_csv(RAW_PATH / "reports.csv")
users = pd.read_csv(RAW_PATH / "users.csv")
report_pages = pd.read_csv(RAW_PATH / "report_pages.csv")
dates = pd.read_csv(RAW_PATH / "dates.csv", parse_dates=["date", "week_start_date"])
report_views = pd.read_csv(RAW_PATH / "report_views.csv", parse_dates=["date"])
report_page_views = pd.read_csv(RAW_PATH / "report_page_views.csv", parse_dates=["timestamp", "date"])
report_load_times = pd.read_csv(RAW_PATH / "report_load_times.csv", parse_dates=["timestamp", "date"])

print("Loaded raw tables successfully.")

Loaded raw tables successfully.


In [3]:
print("reports:", reports.shape)
print("users:", users.shape)
print("report_pages:", report_pages.shape)
print("dates:", dates.shape)
print("report_views:", report_views.shape)
print("report_page_views:", report_page_views.shape)
print("report_load_times:", report_load_times.shape)

reports: (30, 5)
users: (200, 3)
report_pages: (150, 3)
dates: (455, 5)
report_views: (411006, 8)
report_page_views: (879149, 8)
report_load_times: (411006, 9)


# Build cleaned dimensions

Dimension tables provide the descriptive context for the fact tables.

In this model we create:

- `dim_date`
- `dim_user`
- `dim_report`
- `dim_page`

Each dimension should have:

- a clear grain
- stable keys
- no unnecessary duplication


## Build `dim_date`

`dim_date` provides calendar attributes for filtering, grouping, and time series analysis.

### Grain
One row per calendar date.


In [4]:
dim_date = dates.copy()

dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)

dim_date = dim_date[[
    "date_key",
    "date",
    "day_of_week",
    "week_start_date",
    "month",
    "is_weekend"
]].drop_duplicates().sort_values("date").reset_index(drop=True)

dim_date.head()

,date_key,date,day_of_week,week_start_date,month,is_weekend
0,20250101,2025-01-01,Wednesday,2024-12-30,2025-01,False
1,20250102,2025-01-02,Thursday,2024-12-30,2025-01,False
2,20250103,2025-01-03,Friday,2024-12-30,2025-01,False
3,20250104,2025-01-04,Saturday,2024-12-30,2025-01,True
4,20250105,2025-01-05,Sunday,2024-12-30,2025-01,True


## Build `dim_user`

`dim_user` stores the user reference information used across the usage facts.

### Grain
One row per user.


In [5]:
dim_user = users.copy()

dim_user = dim_user.drop_duplicates(subset=["user_key"]).reset_index(drop=True)

dim_user.head()

,user_key,user_id,unique_user
0,UK_0001,user001@masegoinc.com,User 001
1,UK_0002,user002@masegoinc.com,User 002
2,UK_0003,user003@masegoinc.com,User 003
3,UK_0004,user004@masegoinc.com,User 004
4,UK_0005,user005@masegoinc.com,User 005


## Build `dim_report`

`dim_report` stores report-level metadata.

### Grain
One row per report.


In [6]:
dim_report = reports.copy()

dim_report = dim_report.drop_duplicates(subset=["report_id"]).reset_index(drop=True)

dim_report.head()

,report_id,report_name,workspace_id,report_type,is_usage_metrics_report
0,R_001,Report_001,WS_004,Report,False
1,R_002,Report_002,WS_005,Report,False
2,R_003,Report_003,WS_003,Paginated,False
3,R_004,Report_004,WS_005,Report,True
4,R_005,Report_005,WS_005,Report,False


## Build `dim_page`

`dim_page` stores page-level metadata for page-capable reports.

### Grain
One row per page within a report.

We create a surrogate key called `page_key` because page identity is naturally a composite business key of:

`report_id + section_id`

A surrogate key makes downstream joins simpler and cleaner.


In [7]:
dim_page = report_pages.copy()

dim_page = dim_page.drop_duplicates(subset=["report_id", "section_id"]).reset_index(drop=True)
dim_page["page_key"] = range(1, len(dim_page) + 1)

dim_page = dim_page[[
    "page_key",
    "report_id",
    "section_id",
    "section_name"
]]

dim_page.head()

,page_key,report_id,section_id,section_name
0,1,R_001,R_001_P1,Page 1
1,2,R_001,R_001_P2,Page 2
2,3,R_001,R_001_P3,Page 3
3,4,R_001,R_001_P4,Page 4
4,5,R_001,R_001_P5,Page 5


## Quick dimension checks

Before moving to the facts, it is good practice to confirm that the primary keys behave as expected.


In [8]:
print("dim_date unique date_key:", dim_date["date_key"].is_unique)
print("dim_user unique user_key:", dim_user["user_key"].is_unique)
print("dim_report unique report_id:", dim_report["report_id"].is_unique)
print("dim_page unique page_key:", dim_page["page_key"].is_unique)

dim_date unique date_key: True
dim_user unique user_key: True
dim_report unique report_id: True
dim_page unique page_key: True


# Build cleaned facts

Fact tables store measurable usage and performance events.

In this model we create:

- `fact_report_views`
- `fact_page_views`
- `fact_report_loads`

These facts will be linked back to the dimensions using keys.


## Build `fact_report_views`

This is the main report-level behavioral fact.

### Grain
One row per:

`date x report x user x consumption_method x distribution_method`

This fact stores the measure `view_count`.


In [9]:
fact_report_views = report_views.copy()

fact_report_views = fact_report_views.merge(
    dim_date[["date", "date_key"]],
    on="date",
    how="left"
)

fact_report_views = fact_report_views[[
    "date_key",
    "report_id",
    "user_key",
    "consumption_method",
    "distribution_method",
    "view_count"
]].reset_index(drop=True)

fact_report_views.head()

,date_key,report_id,user_key,consumption_method,distribution_method,view_count
0,20250101,R_001,UK_0002,Web,Direct,1
1,20250101,R_001,UK_0004,Web,App,3
2,20250101,R_001,UK_0012,Mobile,App,2
3,20250101,R_001,UK_0019,Web,Direct,1
4,20250101,R_001,UK_0023,Web,Direct,1


## Build `fact_page_views`

This fact stores page-level interaction behavior.

### Grain
For this first version, one row per page-view event.

We map `section_id` to `page_key` so the fact joins cleanly to `dim_page`.


In [10]:
fact_page_views = report_page_views.copy()

fact_page_views = fact_page_views.merge(
    dim_date[["date", "date_key"]],
    on="date",
    how="left"
)

fact_page_views = fact_page_views.merge(
    dim_page[["page_key", "section_id"]],
    on="section_id",
    how="left"
)

fact_page_views = fact_page_views[[
    "date_key",
    "report_id",
    "page_key",
    "user_key",
    "client",
    "session_source",
    "page_view_count"
]].reset_index(drop=True)

fact_page_views.head()

,date_key,report_id,page_key,user_key,client,session_source,page_view_count
0,20250101,R_001,6,UK_0002,Browser,Direct,1
1,20250101,R_001,8,UK_0002,Browser,Direct,1
2,20250101,R_001,7,UK_0004,Browser,App,1
3,20250101,R_001,8,UK_0012,Browser,App,1
4,20250101,R_001,3,UK_0012,Browser,Direct,1


## Build `fact_report_loads`

This fact stores performance telemetry for report load events.

### Grain
One row per report-load event.

This fact stores the measure `load_time_ms`.


In [11]:
fact_report_loads = report_load_times.copy()

fact_report_loads = fact_report_loads.merge(
    dim_date[["date", "date_key"]],
    on="date",
    how="left"
)

fact_report_loads = fact_report_loads[[
    "date_key",
    "report_id",
    "user_key",
    "browser",
    "client",
    "country",
    "load_time_ms"
]].reset_index(drop=True)

fact_report_loads.head()

,date_key,report_id,user_key,browser,client,country,load_time_ms
0,20250101,R_001,UK_0002,Chrome,Browser,Canada,3937.0
1,20250101,R_001,UK_0004,Chrome,Browser,UK,4902.0
2,20250101,R_001,UK_0012,Edge,MobileApp,Canada,4087.0
3,20250101,R_001,UK_0019,Edge,Browser,South Africa,4040.0
4,20250101,R_001,UK_0023,Firefox,Browser,Canada,4460.0


## Validation checks

This is a lightweight first round of validation to confirm:

- key uniqueness in dimensions
- referential integrity between facts and dimensions
- no unexpected nulls in key fields


In [12]:
print("Dimension key uniqueness")
print("------------------------")
print("dim_date date_key unique:", dim_date["date_key"].is_unique)
print("dim_user user_key unique:", dim_user["user_key"].is_unique)
print("dim_report report_id unique:", dim_report["report_id"].is_unique)
print("dim_page page_key unique:", dim_page["page_key"].is_unique)

Dimension key uniqueness
------------------------
dim_date date_key unique: True
dim_user user_key unique: True
dim_report report_id unique: True
dim_page page_key unique: True


In [13]:
print("Referential integrity checks")
print("----------------------------")
print("fact_report_views.report_id in dim_report:",
      fact_report_views["report_id"].isin(dim_report["report_id"]).all())

print("fact_report_views.user_key in dim_user:",
      fact_report_views["user_key"].isin(dim_user["user_key"]).all())

print("fact_page_views.report_id in dim_report:",
      fact_page_views["report_id"].isin(dim_report["report_id"]).all())

print("fact_page_views.page_key in dim_page:",
      fact_page_views["page_key"].isin(dim_page["page_key"]).all())

print("fact_page_views.user_key in dim_user:",
      fact_page_views["user_key"].isin(dim_user["user_key"]).all())

print("fact_report_loads.report_id in dim_report:",
      fact_report_loads["report_id"].isin(dim_report["report_id"]).all())

print("fact_report_loads.user_key in dim_user:",
      fact_report_loads["user_key"].isin(dim_user["user_key"]).all())

Referential integrity checks
----------------------------
fact_report_views.report_id in dim_report: True
fact_report_views.user_key in dim_user: True
fact_page_views.report_id in dim_report: True
fact_page_views.page_key in dim_page: True
fact_page_views.user_key in dim_user: True
fact_report_loads.report_id in dim_report: True
fact_report_loads.user_key in dim_user: True


In [14]:
print("Null checks in fact keys")
print("------------------------")
print("fact_report_views")
print(fact_report_views[["date_key", "report_id", "user_key"]].isnull().sum())

print("\nfact_page_views")
print(fact_page_views[["date_key", "report_id", "page_key", "user_key"]].isnull().sum())

print("\nfact_report_loads")
print(fact_report_loads[["date_key", "report_id", "user_key"]].isnull().sum())

Null checks in fact keys
------------------------
fact_report_views
date_key     0
report_id    0
user_key     0
dtype: int64

fact_page_views
date_key     0
report_id    0
page_key     0
user_key     0
dtype: int64

fact_report_loads
date_key     0
report_id    0
user_key     0
dtype: int64


## Save processed tables

The processed tables are saved to `data/processed/` as CSV files.

This gives the project a clear separation between:

- raw source-aligned data
- cleaned semantic model data


In [15]:
dim_date.to_csv(PROCESSED_PATH / "dim_date.csv", index=False)
dim_user.to_csv(PROCESSED_PATH / "dim_user.csv", index=False)
dim_report.to_csv(PROCESSED_PATH / "dim_report.csv", index=False)
dim_page.to_csv(PROCESSED_PATH / "dim_page.csv", index=False)

fact_report_views.to_csv(PROCESSED_PATH / "fact_report_views.csv", index=False)
fact_page_views.to_csv(PROCESSED_PATH / "fact_page_views.csv", index=False)
fact_report_loads.to_csv(PROCESSED_PATH / "fact_report_loads.csv", index=False)

print("Processed CSV files saved successfully.")

Processed CSV files saved successfully.


## Summary

At this point, the project now has a clean first version of the semantic model.

### Dimensions created
- `dim_date`
- `dim_user`
- `dim_report`
- `dim_page`

### Facts created
- `fact_report_views`
- `fact_page_views`
- `fact_report_loads`

These processed tables are now ready for the next stage of the project:

- report-level feature engineering
- behavioural metrics
- forecasting feature creation
